In [1]:
import pandas as pd

df = pd.read_csv("ag_news_en_500.csv")
print(df.columns)
print(df["label"].value_counts().sort_index())
df.head()

Index(['text', 'label'], dtype='object')
0    121
1    116
2    124
3    139
Name: label, dtype: int64


,text,label
0,Fan v Fan: Manchester City-Tottenham Hotspur T...,1
1,Paris Tourists Search for Key to 'Da Vinci Cod...,0
2,Net firms: Don't tax VoIP The Spanish-American...,3
3,Dependent species risk extinction The global e...,3
4,EDS Is Charter Member of Siebel BPO Alliance (...,3


In [2]:
label_map = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

df["label_name"] = df["label"].map(label_map)
df.head()

,text,label,label_name
0,Fan v Fan: Manchester City-Tottenham Hotspur T...,1,Sports
1,Paris Tourists Search for Key to 'Da Vinci Cod...,0,World
2,Net firms: Don't tax VoIP The Spanish-American...,3,Sci/Tech
3,Dependent species risk extinction The global e...,3,Sci/Tech
4,EDS Is Charter Member of Siebel BPO Alliance (...,3,Sci/Tech


In [4]:
import re
import time
from openai import OpenAI

client = OpenAI(api_key=" ")

def classify_agnews_en_basic(text: str) -> str:
    prompt = f"""You are a news topic classifier.

Classify the following news text into one of these 4 categories:
World, Sports, Business, Sci/Tech

Respond with only one category name.
Do not explain your answer.

News text:
{text}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content.strip()

    # normalize output
    output_lower = output.lower()

    if "world" in output_lower:
        return "World"
    elif "sports" in output_lower:
        return "Sports"
    elif "business" in output_lower:
        return "Business"
    elif "sci" in output_lower or "tech" in output_lower:
        return "Sci/Tech"
    else:
        return output

In [5]:
for i in range(5):
    pred = classify_agnews_en_basic(df.loc[i, "text"])
    print(i, "pred:", pred, "| true:", df.loc[i, "label_name"])
    time.sleep(0.3)

0 pred: Sports | true: Sports
1 pred: World | true: World
2 pred: Business | true: Sci/Tech
3 pred: Sci/Tech | true: Sci/Tech
4 pred: Business | true: Sci/Tech


In [6]:
if "pred_en_basic" not in df.columns:
    df["pred_en_basic"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_en_basic"], str) and df.loc[i, "pred_en_basic"].strip():
        continue

    print(f"AGNews En-Basic {i+1}/{len(df)}")

    df.loc[i, "pred_en_basic"] = classify_agnews_en_basic(df.loc[i, "text"])

    if (i + 1) % 25 == 0:
        df.to_csv("agnews_en_basic_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("agnews_en_basic_final.csv", index=False)
print("English En-Basic finished")

AGNews En-Basic 1/500
AGNews En-Basic 2/500
AGNews En-Basic 3/500
AGNews En-Basic 4/500
AGNews En-Basic 5/500
AGNews En-Basic 6/500
AGNews En-Basic 7/500
AGNews En-Basic 8/500
AGNews En-Basic 9/500
AGNews En-Basic 10/500
AGNews En-Basic 11/500
AGNews En-Basic 12/500
AGNews En-Basic 13/500
AGNews En-Basic 14/500
AGNews En-Basic 15/500
AGNews En-Basic 16/500
AGNews En-Basic 17/500
AGNews En-Basic 18/500
AGNews En-Basic 19/500
AGNews En-Basic 20/500
AGNews En-Basic 21/500
AGNews En-Basic 22/500
AGNews En-Basic 23/500
AGNews En-Basic 24/500
AGNews En-Basic 25/500
AGNews En-Basic 26/500
AGNews En-Basic 27/500
AGNews En-Basic 28/500
AGNews En-Basic 29/500
AGNews En-Basic 30/500
AGNews En-Basic 31/500
AGNews En-Basic 32/500
AGNews En-Basic 33/500
AGNews En-Basic 34/500
AGNews En-Basic 35/500
AGNews En-Basic 36/500
AGNews En-Basic 37/500
AGNews En-Basic 38/500
AGNews En-Basic 39/500
AGNews En-Basic 40/500
AGNews En-Basic 41/500
AGNews En-Basic 42/500
AGNews En-Basic 43/500
AGNews En-Basic 44/5

AGNews En-Basic 347/500
AGNews En-Basic 348/500
AGNews En-Basic 349/500
AGNews En-Basic 350/500
AGNews En-Basic 351/500
AGNews En-Basic 352/500
AGNews En-Basic 353/500
AGNews En-Basic 354/500
AGNews En-Basic 355/500
AGNews En-Basic 356/500
AGNews En-Basic 357/500
AGNews En-Basic 358/500
AGNews En-Basic 359/500
AGNews En-Basic 360/500
AGNews En-Basic 361/500
AGNews En-Basic 362/500
AGNews En-Basic 363/500
AGNews En-Basic 364/500
AGNews En-Basic 365/500
AGNews En-Basic 366/500
AGNews En-Basic 367/500
AGNews En-Basic 368/500
AGNews En-Basic 369/500
AGNews En-Basic 370/500
AGNews En-Basic 371/500
AGNews En-Basic 372/500
AGNews En-Basic 373/500
AGNews En-Basic 374/500
AGNews En-Basic 375/500
AGNews En-Basic 376/500
AGNews En-Basic 377/500
AGNews En-Basic 378/500
AGNews En-Basic 379/500
AGNews En-Basic 380/500
AGNews En-Basic 381/500
AGNews En-Basic 382/500
AGNews En-Basic 383/500
AGNews En-Basic 384/500
AGNews En-Basic 385/500
AGNews En-Basic 386/500
AGNews En-Basic 387/500
AGNews En-Basic 

In [7]:
from sklearn.metrics import accuracy_score, classification_report

df_eval = pd.read_csv("agnews_en_basic_final.csv")

print("Accuracy:", accuracy_score(df_eval["label_name"], df_eval["pred_en_basic"]))
print(classification_report(df_eval["label_name"], df_eval["pred_en_basic"]))

Accuracy: 0.886
              precision    recall  f1-score   support

    Business       0.76      0.93      0.83       124
    Sci/Tech       0.95      0.76      0.84       139
      Sports       0.97      0.99      0.98       116
       World       0.91      0.88      0.90       121

    accuracy                           0.89       500
   macro avg       0.90      0.89      0.89       500
weighted avg       0.90      0.89      0.89       500



In [8]:
import pandas as pd

df = pd.read_csv("ag_news_kz_500.csv")
print(df.columns)

Index(['text', 'label', 'text_kz'], dtype='object')


In [9]:
label_map = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

df["label_name"] = df["label"].map(label_map)

In [10]:
def classify_agnews_native_basic(text_kz: str) -> str:
    prompt = f"""Сен жаңалық мәтінінің тақырыбын анықтайтын сарапшысың.

Төмендегі жаңалық мәтінін 4 санаттың біріне жатқыз:
World, Sports, Business, Sci/Tech

Тек бір санат атауымен жауап бер.
Жауапқа түсіндірме қоспа.

Жаңалық мәтіні:
{text_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content.strip()
    output_lower = output.lower()

    if "world" in output_lower:
        return "World"
    elif "sports" in output_lower:
        return "Sports"
    elif "business" in output_lower:
        return "Business"
    elif "sci" in output_lower or "tech" in output_lower:
        return "Sci/Tech"
    else:
        return output

In [11]:
for i in range(5):
    pred = classify_agnews_native_basic(df.loc[i, "text_kz"])
    print(i, "pred:", pred, "| true:", df.loc[i, "label_name"])
    time.sleep(0.3)

0 pred: Sports | true: Sports
1 pred: World | true: World
2 pred: Business | true: Sci/Tech
3 pred: Sci/Tech | true: Sci/Tech
4 pred: Business | true: Sci/Tech


In [12]:
if "pred_native_basic" not in df.columns:
    df["pred_native_basic"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_native_basic"], str) and df.loc[i, "pred_native_basic"].strip():
        continue

    print(f"AGNews Native-Basic {i+1}/{len(df)}")

    df.loc[i, "pred_native_basic"] = classify_agnews_native_basic(df.loc[i, "text_kz"])

    if (i + 1) % 25 == 0:
        df.to_csv("agnews_native_basic_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("agnews_native_basic_final.csv", index=False)
print("Kazakh Native-Basic finished")

AGNews Native-Basic 1/500
AGNews Native-Basic 2/500
AGNews Native-Basic 3/500
AGNews Native-Basic 4/500
AGNews Native-Basic 5/500
AGNews Native-Basic 6/500
AGNews Native-Basic 7/500
AGNews Native-Basic 8/500
AGNews Native-Basic 9/500
AGNews Native-Basic 10/500
AGNews Native-Basic 11/500
AGNews Native-Basic 12/500
AGNews Native-Basic 13/500
AGNews Native-Basic 14/500
AGNews Native-Basic 15/500
AGNews Native-Basic 16/500
AGNews Native-Basic 17/500
AGNews Native-Basic 18/500
AGNews Native-Basic 19/500
AGNews Native-Basic 20/500
AGNews Native-Basic 21/500
AGNews Native-Basic 22/500
AGNews Native-Basic 23/500
AGNews Native-Basic 24/500
AGNews Native-Basic 25/500
AGNews Native-Basic 26/500
AGNews Native-Basic 27/500
AGNews Native-Basic 28/500
AGNews Native-Basic 29/500
AGNews Native-Basic 30/500
AGNews Native-Basic 31/500
AGNews Native-Basic 32/500
AGNews Native-Basic 33/500
AGNews Native-Basic 34/500
AGNews Native-Basic 35/500
AGNews Native-Basic 36/500
AGNews Native-Basic 37/500
AGNews Nat

AGNews Native-Basic 298/500
AGNews Native-Basic 299/500
AGNews Native-Basic 300/500
AGNews Native-Basic 301/500
AGNews Native-Basic 302/500
AGNews Native-Basic 303/500
AGNews Native-Basic 304/500
AGNews Native-Basic 305/500
AGNews Native-Basic 306/500
AGNews Native-Basic 307/500
AGNews Native-Basic 308/500
AGNews Native-Basic 309/500
AGNews Native-Basic 310/500
AGNews Native-Basic 311/500
AGNews Native-Basic 312/500
AGNews Native-Basic 313/500
AGNews Native-Basic 314/500
AGNews Native-Basic 315/500
AGNews Native-Basic 316/500
AGNews Native-Basic 317/500
AGNews Native-Basic 318/500
AGNews Native-Basic 319/500
AGNews Native-Basic 320/500
AGNews Native-Basic 321/500
AGNews Native-Basic 322/500
AGNews Native-Basic 323/500
AGNews Native-Basic 324/500
AGNews Native-Basic 325/500
AGNews Native-Basic 326/500
AGNews Native-Basic 327/500
AGNews Native-Basic 328/500
AGNews Native-Basic 329/500
AGNews Native-Basic 330/500
AGNews Native-Basic 331/500
AGNews Native-Basic 332/500
AGNews Native-Basic 

In [13]:
from sklearn.metrics import accuracy_score, classification_report

df_eval = pd.read_csv("agnews_native_basic_final.csv")

print("Accuracy:", accuracy_score(df_eval["label_name"], df_eval["pred_native_basic"]))
print(classification_report(df_eval["label_name"], df_eval["pred_native_basic"]))

Accuracy: 0.87
              precision    recall  f1-score   support

    Business       0.72      0.93      0.81       124
    Sci/Tech       0.94      0.71      0.81       139
      Sports       0.97      0.99      0.98       116
       World       0.91      0.88      0.89       121

    accuracy                           0.87       500
   macro avg       0.88      0.88      0.87       500
weighted avg       0.88      0.87      0.87       500



In [14]:
def classify_agnews_en_kz(text_kz: str) -> str:
    prompt = f"""You are a news topic classifier.

Classify the following news text into one of these categories:
World, Sports, Business, Sci/Tech

Respond with only one category name.
Do not explain your answer.

News text:
{text_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content.strip().lower()

    if "world" in output:
        return "World"
    elif "sports" in output:
        return "Sports"
    elif "business" in output:
        return "Business"
    elif "sci" in output or "tech" in output:
        return "Sci/Tech"
    else:
        return output

In [15]:
for i in range(5):
    pred = classify_agnews_en_kz(df.loc[i, "text_kz"])
    print(i, "pred:", pred, "| true:", df.loc[i, "label_name"])
    time.sleep(0.3)

0 pred: Sports | true: Sports
1 pred: World | true: World
2 pred: Business | true: Sci/Tech
3 pred: Sci/Tech | true: Sci/Tech
4 pred: Business | true: Sci/Tech


In [16]:
if "pred_en_kz" not in df.columns:
    df["pred_en_kz"] = ""

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_en_kz"], str) and df.loc[i, "pred_en_kz"].strip():
        continue

    print(f"AGNews EN prompt + KZ input {i+1}/{len(df)}")

    df.loc[i, "pred_en_kz"] = classify_agnews_en_kz(df.loc[i, "text_kz"])

    if (i + 1) % 25 == 0:
        df.to_csv("agnews_en_kz_progress.csv", index=False)

    time.sleep(0.3)

df.to_csv("agnews_en_kz_final.csv", index=False)
print("Finished EN prompt + KZ input")

AGNews EN prompt + KZ input 1/500
AGNews EN prompt + KZ input 2/500
AGNews EN prompt + KZ input 3/500
AGNews EN prompt + KZ input 4/500
AGNews EN prompt + KZ input 5/500
AGNews EN prompt + KZ input 6/500
AGNews EN prompt + KZ input 7/500
AGNews EN prompt + KZ input 8/500
AGNews EN prompt + KZ input 9/500
AGNews EN prompt + KZ input 10/500
AGNews EN prompt + KZ input 11/500
AGNews EN prompt + KZ input 12/500
AGNews EN prompt + KZ input 13/500
AGNews EN prompt + KZ input 14/500
AGNews EN prompt + KZ input 15/500
AGNews EN prompt + KZ input 16/500
AGNews EN prompt + KZ input 17/500
AGNews EN prompt + KZ input 18/500
AGNews EN prompt + KZ input 19/500
AGNews EN prompt + KZ input 20/500
AGNews EN prompt + KZ input 21/500
AGNews EN prompt + KZ input 22/500
AGNews EN prompt + KZ input 23/500
AGNews EN prompt + KZ input 24/500
AGNews EN prompt + KZ input 25/500
AGNews EN prompt + KZ input 26/500
AGNews EN prompt + KZ input 27/500
AGNews EN prompt + KZ input 28/500
AGNews EN prompt + KZ input 2

AGNews EN prompt + KZ input 232/500
AGNews EN prompt + KZ input 233/500
AGNews EN prompt + KZ input 234/500
AGNews EN prompt + KZ input 235/500
AGNews EN prompt + KZ input 236/500
AGNews EN prompt + KZ input 237/500
AGNews EN prompt + KZ input 238/500
AGNews EN prompt + KZ input 239/500
AGNews EN prompt + KZ input 240/500
AGNews EN prompt + KZ input 241/500
AGNews EN prompt + KZ input 242/500
AGNews EN prompt + KZ input 243/500
AGNews EN prompt + KZ input 244/500
AGNews EN prompt + KZ input 245/500
AGNews EN prompt + KZ input 246/500
AGNews EN prompt + KZ input 247/500
AGNews EN prompt + KZ input 248/500
AGNews EN prompt + KZ input 249/500
AGNews EN prompt + KZ input 250/500
AGNews EN prompt + KZ input 251/500
AGNews EN prompt + KZ input 252/500
AGNews EN prompt + KZ input 253/500
AGNews EN prompt + KZ input 254/500
AGNews EN prompt + KZ input 255/500
AGNews EN prompt + KZ input 256/500
AGNews EN prompt + KZ input 257/500
AGNews EN prompt + KZ input 258/500
AGNews EN prompt + KZ input 

AGNews EN prompt + KZ input 460/500
AGNews EN prompt + KZ input 461/500
AGNews EN prompt + KZ input 462/500
AGNews EN prompt + KZ input 463/500
AGNews EN prompt + KZ input 464/500
AGNews EN prompt + KZ input 465/500
AGNews EN prompt + KZ input 466/500
AGNews EN prompt + KZ input 467/500
AGNews EN prompt + KZ input 468/500
AGNews EN prompt + KZ input 469/500
AGNews EN prompt + KZ input 470/500
AGNews EN prompt + KZ input 471/500
AGNews EN prompt + KZ input 472/500
AGNews EN prompt + KZ input 473/500
AGNews EN prompt + KZ input 474/500
AGNews EN prompt + KZ input 475/500
AGNews EN prompt + KZ input 476/500
AGNews EN prompt + KZ input 477/500
AGNews EN prompt + KZ input 478/500
AGNews EN prompt + KZ input 479/500
AGNews EN prompt + KZ input 480/500
AGNews EN prompt + KZ input 481/500
AGNews EN prompt + KZ input 482/500
AGNews EN prompt + KZ input 483/500
AGNews EN prompt + KZ input 484/500
AGNews EN prompt + KZ input 485/500
AGNews EN prompt + KZ input 486/500
AGNews EN prompt + KZ input 

In [17]:
df_eval = pd.read_csv("agnews_en_kz_final.csv")

print("Accuracy:", accuracy_score(df_eval["label_name"], df_eval["pred_en_kz"]))
print(classification_report(df_eval["label_name"], df_eval["pred_en_kz"]))

Accuracy: 0.874
              precision    recall  f1-score   support

    Business       0.75      0.92      0.83       124
    Sci/Tech       0.94      0.72      0.82       139
      Sports       0.97      0.99      0.98       116
       World       0.88      0.89      0.89       121

    accuracy                           0.87       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.88      0.87      0.87       500



In [1]:
import pandas as pd

df = pd.read_csv("ag_news_kz_500.csv")
print(df.columns)

Index(['text', 'label', 'text_kz'], dtype='object')


In [2]:
df_small = df.sample(n=100, random_state=42).reset_index(drop=True)
df_small["pred_translate_en"] = ""

In [3]:
def classify_agnews_translate_en(text_kz: str) -> str:

    prompt = f"""You are a news topic classifier.

Step 1: Translate the following news text into English.
Step 2: Classify it into one of these categories:
World, Sports, Business, Sci/Tech

Respond with ONLY the final category name.
Do not show the translation.

Text:
{text_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content.strip().lower()

    if "world" in output:
        return "World"
    elif "sports" in output:
        return "Sports"
    elif "business" in output:
        return "Business"
    elif "sci" in output or "tech" in output:
        return "Sci/Tech"
    else:
        return output

In [6]:
label_map = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

df_small["label_name"] = df_small["label"].map(label_map)

In [7]:
for i in range(5):
    pred = classify_agnews_translate_en(df_small.loc[i, "text_kz"])
    print(i, pred, "|", df_small.loc[i, "label_name"])

0 Sports | Sports
1 Sci/Tech | Sci/Tech
2 Business | Business
3 Sports | Sports
4 World | World


In [8]:
import time

for i in range(len(df_small)):

    print(f"Translate-En {i+1}/100")

    df_small.loc[i, "pred_translate_en"] = classify_agnews_translate_en(
        df_small.loc[i, "text_kz"]
    )

    if (i+1) % 20 == 0:
        df_small.to_csv("agnews_translate_en_100_progress.csv", index=False)

    time.sleep(0.3)

df_small.to_csv("agnews_translate_en_100_final.csv", index=False)

print("Translate-En finished")

Translate-En 1/100
Translate-En 2/100
Translate-En 3/100
Translate-En 4/100
Translate-En 5/100
Translate-En 6/100
Translate-En 7/100
Translate-En 8/100
Translate-En 9/100
Translate-En 10/100
Translate-En 11/100
Translate-En 12/100
Translate-En 13/100
Translate-En 14/100
Translate-En 15/100
Translate-En 16/100
Translate-En 17/100
Translate-En 18/100
Translate-En 19/100
Translate-En 20/100
Translate-En 21/100
Translate-En 22/100
Translate-En 23/100
Translate-En 24/100
Translate-En 25/100
Translate-En 26/100
Translate-En 27/100
Translate-En 28/100
Translate-En 29/100
Translate-En 30/100
Translate-En 31/100
Translate-En 32/100
Translate-En 33/100
Translate-En 34/100
Translate-En 35/100
Translate-En 36/100
Translate-En 37/100
Translate-En 38/100
Translate-En 39/100
Translate-En 40/100
Translate-En 41/100
Translate-En 42/100
Translate-En 43/100
Translate-En 44/100
Translate-En 45/100
Translate-En 46/100
Translate-En 47/100
Translate-En 48/100
Translate-En 49/100
Translate-En 50/100
Translate

In [9]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(df_small["label_name"], df_small["pred_translate_en"]))
print(classification_report(df_small["label_name"], df_small["pred_translate_en"]))

Accuracy: 0.85
              precision    recall  f1-score   support

    Business       0.70      0.84      0.76        25
    Sci/Tech       0.95      0.70      0.81        30
      Sports       1.00      1.00      1.00        29
       World       0.74      0.88      0.80        16

    accuracy                           0.85       100
   macro avg       0.85      0.85      0.84       100
weighted avg       0.87      0.85      0.85       100

